# Qwen3 secondary embedding index — resumable T4×2

Attach the **private** dataset produced by `prepare_qwen3_embedding_kaggle.py`, enable Internet and two T4 GPUs. The default `0.6B` run is the fastest full-corpus branch. `PARTITION_COUNT`/`PARTITION_INDEX` allow multiple authorized Kaggle workers to split shards without overlap: for two workers, use index 0 and 1. Change `MODEL_KEY` to `4B` only for a proven, affordable staged run. Different model spaces must never be merged.

Documents receive no instruction. VI/EN/ZH query variants use one fixed multilingual-medical retrieval instruction. Output is FP16 shards for transfer; the local builder validates and converts them to a standalone FP32 FAISS FlatIP index.

In [ ]:
!pip install -q "transformers>=4.51.0,<5" "huggingface_hub>=0.30" safetensors

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time
import numpy as np
import pandas as pd
import torch
from huggingface_hub import snapshot_download

MODEL_KEY = "0.6B"  # Use "4B" only for staged/resumed quality run.
# Run the same notebook on authorized independent Kaggle workers/accounts.
# Worker A: PARTITION_INDEX = 0; worker B: PARTITION_INDEX = 1.
PARTITION_COUNT = 2
PARTITION_INDEX = 0
MODEL_SPECS = {
    "0.6B": {
        "repo": "Qwen/Qwen3-Embedding-0.6B",
        "revision": "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3",
        "batch_size": 32,
    },
    "4B": {
        "repo": "Qwen/Qwen3-Embedding-4B",
        "revision": "5cf2132abc99cad020ac570b19d031efec650f2b",
        "batch_size": 8,
    },
}
OUTPUT_DIM = 1024
MAX_LENGTH = 1024
QUERY_INSTRUCTION = "Given a multilingual medical question, retrieve passages that contain information needed to answer the question."

assert torch.cuda.device_count() == 2, f"Expected Kaggle T4 x2, found {torch.cuda.device_count()} GPU(s)"
assert PARTITION_COUNT >= 1 and 0 <= PARTITION_INDEX < PARTITION_COUNT
spec = MODEL_SPECS[MODEL_KEY]
input_manifests = list(Path("/kaggle/input").rglob("input_manifest.json"))
input_manifests = [p for p in input_manifests if "chunks_metadata" in p.read_text(errors="ignore")]
assert len(input_manifests) == 1, f"Expected one embedding input manifest, found: {input_manifests}"
INPUT_DIR = input_manifests[0].parent
INPUT_MANIFEST = json.loads(input_manifests[0].read_text())
assert INPUT_MANIFEST["full_corpus"], "Production run requires a full-corpus input pack"
PARTITION_ITEMS = [
    item for number, item in enumerate(INPUT_MANIFEST["shards"])
    if number % PARTITION_COUNT == PARTITION_INDEX
]
PARTITION_ROWS = sum(int(item["rows"]) for item in PARTITION_ITEMS)
assert PARTITION_ITEMS and PARTITION_ROWS > 0
OUTPUT_DIR = Path(
    f"/kaggle/working/qwen3_embedding_{MODEL_KEY.lower().replace('.', 'p')}_1024d"
    f"_part{PARTITION_INDEX}of{PARTITION_COUNT}"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = Path("/kaggle/working/models") / MODEL_KEY.replace(".", "p")
print({
    "model": spec,
    "partition": f"{PARTITION_INDEX}/{PARTITION_COUNT}",
    "partition_rows": PARTITION_ROWS,
    "partition_shards": len(PARTITION_ITEMS),
    "global_rows": INPUT_MANIFEST["rows"],
    "global_shards": len(INPUT_MANIFEST["shards"]),
    "output": str(OUTPUT_DIR),
})

## Restore compatible completed shards

If a previous Kaggle output dataset is attached, compatible `done/rows/emb` triplets are copied into the working output. Incompatible model, revision, dimension, max length, instruction or input-manifest fingerprints are ignored.

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while block := handle.read(8 * 1024 * 1024):
            digest.update(block)
    return digest.hexdigest()

RUN_CONFIG = {
    "format_version": 1,
    "model": spec["repo"],
    "revision": spec["revision"],
    "output_dim": OUTPUT_DIM,
    "max_length": MAX_LENGTH,
    "query_instruction": QUERY_INSTRUCTION,
    "document_instruction": None,
    "pooling": "last_token_then_slice_then_l2_normalize",
    "input_manifest_sha256": sha256(input_manifests[0]),
    "partition_count": PARTITION_COUNT,
    "partition_index": PARTITION_INDEX,
}
(OUTPUT_DIR / "run_config.json").write_text(json.dumps(RUN_CONFIG, indent=2))

restored = 0
for prior_config_path in Path("/kaggle/input").rglob("run_config.json"):
    if prior_config_path == input_manifests[0]:
        continue
    try:
        if json.loads(prior_config_path.read_text()) != RUN_CONFIG:
            continue
    except Exception:
        continue
    prior = prior_config_path.parent
    for done_path in prior.glob("done_*.json"):
        tag = done_path.stem.removeprefix("done_")
        trio = [done_path, prior / f"rows_{tag}.npy", prior / f"emb_{tag}.npy"]
        if all(path.exists() for path in trio):
            for path in trio:
                target = OUTPUT_DIR / path.name
                if not target.exists():
                    shutil.copy2(path, target)
            restored += 1
print(f"Restored {restored} completed shard(s)")

In [ ]:
snapshot_download(
    repo_id=spec["repo"],
    revision=spec["revision"],
    local_dir=MODEL_DIR,
    ignore_patterns=["*.onnx", "*.gguf"],
)
worker_path = Path("/kaggle/working/qwen_embed_worker.py")
worker_path.write_text('import argparse\nimport hashlib\nimport json\nimport os\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn.functional as F\nfrom transformers import AutoModel, AutoTokenizer\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with open(path, "rb") as handle:\n        while True:\n            block = handle.read(8 * 1024 * 1024)\n            if not block:\n                break\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef last_token_pool(last_hidden_state, attention_mask):\n    if bool((attention_mask[:, -1] == 1).all()):\n        return last_hidden_state[:, -1]\n    sequence_lengths = attention_mask.sum(dim=1) - 1\n    batch_size = last_hidden_state.shape[0]\n    return last_hidden_state[torch.arange(batch_size, device=last_hidden_state.device), sequence_lengths]\n\n\ndef load_model(model_path, gpu):\n    torch.cuda.set_device(gpu)\n    tokenizer = AutoTokenizer.from_pretrained(model_path, padding_side="left", local_files_only=True)\n    model = AutoModel.from_pretrained(\n        model_path,\n        torch_dtype=torch.float16,\n        attn_implementation="sdpa",\n        local_files_only=True,\n    ).eval().to(f"cuda:{gpu}")\n    model.config.use_cache = False\n    return tokenizer, model\n\n\n@torch.inference_mode()\ndef encode(texts, tokenizer, model, batch_size, max_length, output_dim):\n    vectors = np.empty((len(texts), output_dim), dtype=np.float16)\n    # Similar lengths in a batch substantially reduce left-padding work.\n    order = np.argsort(np.fromiter((len(x) for x in texts), dtype=np.int32), kind="stable")\n    for offset in range(0, len(order), batch_size):\n        selected = order[offset:offset + batch_size]\n        batch = [texts[int(index)] for index in selected]\n        tokens = tokenizer(\n            batch,\n            padding=True,\n            truncation=True,\n            max_length=max_length,\n            return_tensors="pt",\n        ).to(model.device)\n        hidden = model(**tokens, use_cache=False).last_hidden_state\n        pooled = last_token_pool(hidden, tokens["attention_mask"])\n        pooled = F.normalize(pooled[:, :output_dim], p=2, dim=1)\n        vectors[selected] = pooled.float().cpu().numpy().astype(np.float16)\n    return vectors\n\n\ndef documents(args, tokenizer, model):\n    tasks = json.loads(Path(args.tasks).read_text())\n    output_dir = Path(args.output)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    for position, task in enumerate(tasks, 1):\n        source = Path(task["path"])\n        tag = source.stem.removeprefix("chunks_")\n        emb_path = output_dir / f"emb_{tag}.npy"\n        rows_path = output_dir / f"rows_{tag}.npy"\n        done_path = output_dir / f"done_{tag}.json"\n        if done_path.exists() and emb_path.exists() and rows_path.exists():\n            print(f"gpu{args.gpu} skip {source.name} ({position}/{len(tasks)})", flush=True)\n            continue\n        if sha256(source) != task["sha256"]:\n            raise RuntimeError(f"Input checksum mismatch: {source}")\n        frame = pd.read_parquet(source, columns=["row_id", "chunk_id", "chunk_text", "is_blank"])\n        rows = frame.row_id.to_numpy(dtype=np.int64)\n        if not np.array_equal(rows, frame.chunk_id.to_numpy(dtype=np.int64)):\n            raise RuntimeError(f"row_id/chunk_id mismatch: {source}")\n        texts = frame.chunk_text.astype(str).tolist()\n        blank = frame.is_blank.to_numpy(dtype=bool)\n        if any((not value.strip()) != bool(flag) for value, flag in zip(texts, blank)):\n            raise RuntimeError(f"Blank-text flag mismatch: {source}")\n        started = time.time()\n        vectors = np.zeros((len(texts), args.output_dim), dtype=np.float16)\n        keep = np.flatnonzero(~blank)\n        if len(keep):\n            vectors[keep] = encode(\n                [texts[int(index)] for index in keep],\n                tokenizer,\n                model,\n                args.batch_size,\n                args.max_length,\n                args.output_dim,\n            )\n        np.save(emb_path, vectors, allow_pickle=False)\n        np.save(rows_path, rows, allow_pickle=False)\n        done = {\n            "source": source.name,\n            "source_sha256": task["sha256"],\n            "rows": len(rows),\n            "row_min": int(rows.min()),\n            "row_max": int(rows.max()),\n            "embedding_shape": list(vectors.shape),\n            "embedding_dtype": str(vectors.dtype),\n            "blank_rows": int(blank.sum()),\n            "embedding_sha256": sha256(emb_path),\n            "rows_sha256": sha256(rows_path),\n            "seconds": round(time.time() - started, 3),\n        }\n        done_path.write_text(json.dumps(done, indent=2))\n        print(f"gpu{args.gpu} wrote {source.name}: {len(rows):,} rows in {done[\'seconds\']:.1f}s", flush=True)\n\n\ndef queries(args, tokenizer, model):\n    frame = pd.read_parquet(args.query_file)\n    instruction = args.instruction\n    result = {"query_id": frame.id.to_numpy(dtype=np.int64)}\n    original = frame["query"].fillna("").astype(str)\n    for language, column in (("vi", "query"), ("en", "query_en"), ("zh", "query_zh")):\n        values = frame[column].fillna("").astype(str)\n        values = values.where(values.str.strip().ne(""), original)\n        prompted = [f"Instruct: {instruction}\\nQuery:{value}" for value in values]\n        result[language] = encode(\n            prompted, tokenizer, model, args.batch_size, args.max_length, args.output_dim\n        )\n    np.savez(Path(args.output) / "query_embeddings.npz", **result)\n\n\ndef benchmark(args, tokenizer, model):\n    frame = pd.read_parquet(args.benchmark_file, columns=["chunk_text", "is_blank"])\n    frame = frame.loc[~frame.is_blank]\n    count = min(args.benchmark_rows, len(frame))\n    indices = np.linspace(0, len(frame) - 1, count, dtype=np.int64)\n    texts = frame.iloc[indices].chunk_text.astype(str).tolist()\n    torch.cuda.synchronize(args.gpu)\n    started = time.time()\n    vectors = encode(texts, tokenizer, model, args.batch_size, args.max_length, args.output_dim)\n    torch.cuda.synchronize(args.gpu)\n    seconds = time.time() - started\n    result = {\n        "rows": len(texts),\n        "seconds": seconds,\n        "rows_per_second_one_gpu": len(texts) / seconds,\n        "vector_shape": list(vectors.shape),\n    }\n    (Path(args.output) / "benchmark.json").write_text(json.dumps(result, indent=2))\n    print(json.dumps(result), flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--mode", choices=["documents", "queries", "benchmark"], required=True)\n    parser.add_argument("--gpu", type=int, required=True)\n    parser.add_argument("--model-path", required=True)\n    parser.add_argument("--output", required=True)\n    parser.add_argument("--tasks")\n    parser.add_argument("--query-file")\n    parser.add_argument("--benchmark-file")\n    parser.add_argument("--benchmark-rows", type=int, default=1024)\n    parser.add_argument("--instruction")\n    parser.add_argument("--batch-size", type=int, required=True)\n    parser.add_argument("--max-length", type=int, required=True)\n    parser.add_argument("--output-dim", type=int, required=True)\n    args = parser.parse_args()\n    tokenizer, model = load_model(args.model_path, args.gpu)\n    if args.mode == "documents":\n        documents(args, tokenizer, model)\n    elif args.mode == "queries":\n        queries(args, tokenizer, model)\n    else:\n        benchmark(args, tokenizer, model)\n\n\nif __name__ == "__main__":\n    main()')
print("Pinned model downloaded:", spec["repo"], spec["revision"])

## Throughput pilot

The pilot encodes a length-spread sample on one GPU and reports a rough two-GPU full-corpus ETA. Model loading is excluded. Treat the estimate as directional because shard language/length composition varies.

In [ ]:
benchmark_source = INPUT_DIR / PARTITION_ITEMS[0]["file"]
benchmark_command = [
    sys.executable, str(worker_path), "--mode", "benchmark", "--gpu", "0",
    "--model-path", str(MODEL_DIR), "--output", str(OUTPUT_DIR),
    "--benchmark-file", str(benchmark_source), "--benchmark-rows", "1024",
    "--batch-size", str(spec["batch_size"]), "--max-length", str(MAX_LENGTH),
    "--output-dim", str(OUTPUT_DIM),
]
subprocess.run(benchmark_command, check=True)
benchmark = json.loads((OUTPUT_DIR / "benchmark.json").read_text())
partition_eta_hours = PARTITION_ROWS / (2 * benchmark["rows_per_second_one_gpu"]) / 3600
global_four_gpu_eta_hours = INPUT_MANIFEST["rows"] / (2 * PARTITION_COUNT * benchmark["rows_per_second_one_gpu"]) / 3600
print({
    **benchmark,
    "rough_this_partition_eta_hours": round(partition_eta_hours, 2),
    "rough_all_partitions_parallel_eta_hours": round(global_four_gpu_eta_hours, 2),
})

## Two-GPU document embedding

Each assigned input shard is atomic. Completed shards are skipped on rerun. Logs are written separately per GPU, and failures stop manifest creation. Every partition produces its own partial manifest; the local builder combines all partition outputs and requires exact global row coverage.

In [ ]:
tasks = []
for item in PARTITION_ITEMS:
    source = INPUT_DIR / item["file"]
    assert source.exists(), source
    tasks.append({"path": str(source), "sha256": item["sha256"], "rows": item["rows"]})

pending = []
for task in tasks:
    tag = Path(task["path"]).stem.removeprefix("chunks_")
    if not all((OUTPUT_DIR / f"{prefix}_{tag}.{suffix}").exists() for prefix, suffix in (("done", "json"), ("rows", "npy"), ("emb", "npy"))):
        pending.append(task)
print(f"Pending {len(pending)}/{len(tasks)} shards")

task_files = []
processes = []
for gpu in range(2):
    assigned = pending[gpu::2]
    task_path = Path(f"/kaggle/working/tasks_gpu{gpu}.json")
    task_path.write_text(json.dumps(assigned))
    task_files.append(task_path)
    log_path = OUTPUT_DIR / f"worker_gpu{gpu}.log"
    command = [
        sys.executable, str(worker_path), "--mode", "documents", "--gpu", str(gpu),
        "--model-path", str(MODEL_DIR), "--output", str(OUTPUT_DIR), "--tasks", str(task_path),
        "--batch-size", str(spec["batch_size"]), "--max-length", str(MAX_LENGTH),
        "--output-dim", str(OUTPUT_DIM),
    ]
    log_handle = open(log_path, "a", buffering=1)
    processes.append((subprocess.Popen(command, stdout=log_handle, stderr=subprocess.STDOUT), log_handle, log_path))

while any(process.poll() is None for process, _, _ in processes):
    complete = len(list(OUTPUT_DIR.glob("done_*.json")))
    print(f"{time.strftime('%H:%M:%S')} complete {complete}/{len(tasks)}", flush=True)
    time.sleep(60)
for process, handle, log_path in processes:
    handle.close()
    if process.returncode:
        print(log_path.read_text()[-8000:])
        raise RuntimeError(f"Embedding worker failed with code {process.returncode}: {log_path}")
print("All document workers completed")

## Query embeddings and final integrity manifest

In [ ]:
query_command = [
    sys.executable, str(worker_path), "--mode", "queries", "--gpu", "0",
    "--model-path", str(MODEL_DIR), "--output", str(OUTPUT_DIR),
    "--query-file", str(INPUT_DIR / INPUT_MANIFEST["queries"]["file"]),
    "--instruction", QUERY_INSTRUCTION, "--batch-size", str(spec["batch_size"]),
    "--max-length", str(MAX_LENGTH), "--output-dim", str(OUTPUT_DIM),
]
subprocess.run(query_command, check=True)

done = [json.loads(path.read_text()) for path in sorted(OUTPUT_DIR.glob("done_*.json"))]
assert len(done) == len(tasks), (len(done), len(tasks))
assert sum(item["rows"] for item in done) == PARTITION_ROWS
query_path = OUTPUT_DIR / "query_embeddings.npz"
with np.load(query_path, allow_pickle=False) as query_data:
    assert query_data["query_id"].shape == (1200,)
    for language in ("vi", "en", "zh"):
        assert query_data[language].shape == (1200, OUTPUT_DIM)
        assert np.isfinite(query_data[language]).all()

output_manifest = {
    **RUN_CONFIG,
    "rows": PARTITION_ROWS,
    "global_rows": INPUT_MANIFEST["rows"],
    "full_corpus": PARTITION_COUNT == 1,
    "shards": done,
    "query_embeddings": {
        "file": query_path.name,
        "sha256": sha256(query_path),
        "languages": ["vi", "en", "zh"],
        "rows_per_language": 1200,
    },
    "transfer_dtype": "float16",
    "faiss_dtype_after_local_build": "float32",
}
(OUTPUT_DIR / "output_manifest.json").write_text(json.dumps(output_manifest, indent=2))
print(json.dumps({
    "status": "complete",
    "rows": output_manifest["rows"],
    "shards": len(done),
    "output_dir": str(OUTPUT_DIR),
    "manifest_sha256": sha256(OUTPUT_DIR / "output_manifest.json"),
}, indent=2))

## Pack only transfer artifacts

The model, worker scripts, task files, logs and benchmark output are not needed by the local index builder. This cell creates one validated, uncompressed ZIP containing only the manifests, query embeddings, document embeddings and row maps. `ZIP_STORED` avoids wasting time and space trying to recompress near-incompressible FP16 vectors. After validation, the bulky working files are removed so Kaggle exposes only the transfer ZIP.

In [ ]:
import zipfile

required = [
    OUTPUT_DIR / "output_manifest.json",
    OUTPUT_DIR / "run_config.json",
    OUTPUT_DIR / "query_embeddings.npz",
]
for item in done:
    tag = Path(item["source"]).stem.removeprefix("chunks_")
    required.extend([OUTPUT_DIR / f"emb_{tag}.npy", OUTPUT_DIR / f"rows_{tag}.npy"])

assert len(required) == 3 + 2 * len(done)
assert len({path.name for path in required}) == len(required)
assert all(path.is_file() for path in required)
archive_path = Path("/kaggle/working") / f"{OUTPUT_DIR.name}_artifacts.zip"
archive_path.unlink(missing_ok=True)
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
    for path in required:
        archive.write(path, arcname=f"{OUTPUT_DIR.name}/{path.name}")

expected_names = {f"{OUTPUT_DIR.name}/{path.name}" for path in required}
with zipfile.ZipFile(archive_path, "r") as archive:
    assert set(archive.namelist()) == expected_names
    assert archive.testzip() is None
    assert sum(item.file_size for item in archive.infolist()) == sum(path.stat().st_size for path in required)

archive_bytes = archive_path.stat().st_size
archive_sha256 = sha256(archive_path)
shutil.rmtree(OUTPUT_DIR)
shutil.rmtree(MODEL_DIR)
worker_path.unlink(missing_ok=True)
for task_path in task_files:
    task_path.unlink(missing_ok=True)
print(json.dumps({
    "status": "packed_and_cleaned",
    "archive": str(archive_path),
    "files": len(required),
    "bytes": archive_bytes,
    "sha256": archive_sha256,
    "excluded": ["model", "worker", "task files", "logs", "benchmark", "raw output directory"],
}, indent=2))